In [29]:
import pandas as pd
import joblib
import requests
import json
import os
import re

from jsonschema import validate, ValidationError


In [5]:
!pip install openai jsonschema requests

In [2]:
data = pd.read_csv("cleaned_data.csv")

print("Dataset loaded successfully!")
print(data.shape)

Dataset loaded successfully!
(87370, 31)


In [4]:
best_model = joblib.load("best_model (2).pkl")

print("Best model loaded successfully!")

Best model loaded successfully!


In [6]:
print(type(best_model))

<class 'sklearn.pipeline.Pipeline'>


In [7]:
# Show the feature names used by the model
print(best_model.feature_names_in_)

['lead_time' 'arrival_date_year' 'arrival_date_week_number'
 'arrival_date_day_of_month' 'stays_in_weekend_nights'
 'stays_in_week_nights' 'adults' 'children' 'babies' 'is_repeated_guest'
 'previous_cancellations' 'previous_bookings_not_canceled'
 'booking_changes' 'agent' 'days_in_waiting_list'
 'required_car_parking_spaces' 'total_of_special_requests'
 'hotel_Resort Hotel' 'arrival_date_month_August'
 'arrival_date_month_December' 'arrival_date_month_February'
 'arrival_date_month_January' 'arrival_date_month_July'
 'arrival_date_month_June' 'arrival_date_month_March'
 'arrival_date_month_May' 'arrival_date_month_November'
 'arrival_date_month_October' 'arrival_date_month_September' 'meal_FB'
 'meal_HB' 'meal_SC' 'meal_Undefined' 'country_AGO' 'country_AIA'
 'country_ALB' 'country_AND' 'country_ARE' 'country_ARG' 'country_ARM'
 'country_ASM' 'country_ATA' 'country_ATF' 'country_AUS' 'country_AUT'
 'country_AZE' 'country_BDI' 'country_BEL' 'country_BEN' 'country_BFA'
 'country_BGD' 'c

In [8]:
import os
from getpass import getpass

api_key = getpass("Enter your OpenRouter API Key: ")

os.environ["LLM_API_KEY"] = api_key

print("API key stored successfully!")

Enter your OpenRouter API Key: ··········
API key stored successfully!


In [30]:


def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    api_key = os.environ.get("LLM_API_KEY")

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openai/gpt-4.1-mini",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [14]:
print(best_model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(n_estimators=200, random_state=42))])


In [15]:
# Create features exactly like Part 2

X = data.drop(columns=[
    "adr",
    "is_canceled",
    "reservation_status",
    "reservation_status_date"
])

categorical_columns = X.select_dtypes(include=["object"]).columns

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

print("Feature Matrix Shape:", X.shape)

Feature Matrix Shape: (87370, 245)


In [16]:
# Select three hotel booking records
sample_inputs = X.iloc[[0, 1, 2]]

print("Shape:", sample_inputs.shape)

sample_inputs.head()

Shape: (3, 245)


,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,...,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,assigned_room_type_L,assigned_room_type_P,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,342,2015,27,1,0,0,2,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False
1,737,2015,27,1,0,0,2,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False
2,7,2015,27,1,0,1,1,0.0,0,0,...,False,False,False,False,False,False,False,False,True,False


In [17]:
# Predict using the saved model

predictions = best_model.predict(sample_inputs)

probabilities = best_model.predict_proba(sample_inputs)

print("Predicted Classes:")
print(predictions)

print("\nPrediction Probabilities:")
print(probabilities)

Predicted Classes:
[0 0 0]

Prediction Probabilities:
[[0.945 0.055]
 [0.935 0.065]
 [0.955 0.045]]


In [19]:
system_prompt = """
You are an AI assistant.

Your job is to explain hotel booking predictions.

Always return ONLY valid JSON.

Return this format:

{
  "prediction_label":"",
  "confidence_level":"",
  "top_reason":"",
  "second_reason":"",
  "next_step":""
}
"""

In [20]:
# Create the user prompt for the first booking

user_prompt = f"""
Hotel Booking Features:

{sample_inputs.iloc[0].to_dict()}

Prediction:
{predictions[0]}

Prediction Probability:
{probabilities[0][predictions[0]]:.3f}

Explain this prediction.

Return ONLY valid JSON in this format:

{{
  "prediction_label":"",
  "confidence_level":"",
  "top_reason":"",
  "second_reason":"",
  "next_step":""
}}
"""

In [21]:
response = call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=200
)

print(response)

{
  "prediction_label": "Not Canceled",
  "confidence_level": "94.5%",
  "top_reason": "Long lead time of 342 days indicates a planned and committed booking.",
  "second_reason": "Booking is for a Resort Hotel in July with no previous cancellations and no special requests, suggesting a straightforward reservation.",
  "next_step": "Confirm the booking and prepare for guest arrival, ensuring room type C is ready as reserved and assigned."
}


In [22]:
from jsonschema import validate

schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

In [23]:
import json
from jsonschema import validate, ValidationError

try:
    # Convert LLM response to Python dictionary
    explanation = json.loads(response.strip())

    # Validate against the schema
    validate(instance=explanation, schema=schema)

    print("✅ Validation Passed!")
    print("\nParsed JSON:")
    print(explanation)

except json.JSONDecodeError:
    print("❌ Invalid JSON format.")

except ValidationError as e:
    print("❌ Schema Validation Failed!")
    print(e)

✅ Validation Passed!

Parsed JSON:
{'prediction_label': 'Not Canceled', 'confidence_level': '94.5%', 'top_reason': 'Long lead time of 342 days indicates a planned and committed booking.', 'second_reason': 'Booking is for a Resort Hotel in July with no previous cancellations and no special requests, suggesting a straightforward reservation.', 'next_step': 'Confirm the booking and prepare for guest arrival, ensuring room type C is ready as reserved and assigned.'}


In [24]:
import re

def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [25]:
# Test 1 - Contains email (should be blocked)

test_input = "My email is student@gmail.com"

if has_pii(test_input):
    print("Input blocked: PII detected.")
else:
    print("No PII detected.")

Input blocked: PII detected.


In [26]:
# Test 2 - Clean input (should pass)

test_input = "Explain this hotel booking prediction."

if has_pii(test_input):
    print("Input blocked: PII detected.")
else:
    print("No PII detected.")

No PII detected.


In [27]:
import json
from jsonschema import validate, ValidationError

for i in range(3):

    print("=" * 60)
    print(f"Record {i+1}")

    # Create prompt
    user_prompt = f"""
Hotel Booking Features:

{sample_inputs.iloc[i].to_dict()}

Prediction:
{predictions[i]}

Prediction Probability:
{probabilities[i][predictions[i]]:.3f}

Explain this prediction.

Return ONLY valid JSON in this format:

{{
  "prediction_label":"",
  "confidence_level":"",
  "top_reason":"",
  "second_reason":"",
  "next_step":""
}}
"""

    # Guardrail
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        continue

    # LLM Call
    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=0.0,
        max_tokens=200
    )

    print("\nRaw Response:")
    print(response)

    # Validation
    try:
        explanation = json.loads(response.strip())
        validate(instance=explanation, schema=schema)
        print("\n✅ Validation Passed!")

    except json.JSONDecodeError:
        print("❌ Invalid JSON")

    except ValidationError as e:
        print("❌ Schema Validation Failed!")
        print(e)

    print()

Record 1

Raw Response:
{
  "prediction_label": "Not Canceled",
  "confidence_level": "94.5%",
  "top_reason": "Long lead time of 342 days indicates a planned booking, which is less likely to be canceled.",
  "second_reason": "The booking is for a Resort Hotel in July with no previous cancellations and no special requests, suggesting a stable and confirmed reservation.",
  "next_step": "Confirm the booking details with the customer and prepare for the arrival as scheduled."
}

✅ Validation Passed!

Record 2

Raw Response:
{
  "prediction_label": "Not Canceled",
  "confidence_level": "93.5%",
  "top_reason": "Long lead time of 737 days indicates a planned and committed booking.",
  "second_reason": "Direct market segment and distribution channel suggest a reliable booking source.",
  "next_step": "Confirm booking details and prepare for guest arrival."
}

✅ Validation Passed!

Record 3

Raw Response:
{
  "prediction_label": "Not Canceled",
  "confidence_level": "95.5%",
  "top_reason": 

In [28]:
print("Temperature = 0.0")

response_temp0 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_tokens=200
)

print(response_temp0)

print("\n" + "="*60 + "\n")

print("Temperature = 0.7")

response_temp07 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0.7,
    max_tokens=200
)

print(response_temp07)

Temperature = 0.0
{
  "prediction_label": "Not Canceled",
  "confidence_level": "95.5%",
  "top_reason": "Short lead time of 7 days indicates a likely commitment to the booking.",
  "second_reason": "The booking is for a Resort Hotel with a single adult and no previous cancellations, suggesting a low risk of cancellation.",
  "next_step": "Confirm the booking and prepare for guest arrival, ensuring all special requests and room assignments are ready."
}


Temperature = 0.7
{
  "prediction_label": "Not Canceled",
  "confidence_level": "95.5%",
  "top_reason": "Short lead time of 7 days and a single night stay in a resort hotel indicate a planned, likely non-canceled booking.",
  "second_reason": "The booking is direct from a transient customer from Great Britain, which typically shows lower cancellation rates.",
  "next_step": "Confirm booking details with the customer and prepare for a smooth check-in process."
}
